# Session 1: a wheel on camera
**A. Real-world problem.** In films, car wheels and helicopter blades sometimes seem to turn
slowly, stop, or spin the wrong way. The wheel is fine; the camera only looks at it
24 or 30 times a second and our brain joins the dots in the easiest way.

**Plan.** (1) where is the marker in each frame, (2) how far does it seem to move between
frames, (3) turn that into an apparent speed, (4) plot apparent speed against true speed.

In [ ]:
import pathlib

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: how far does the marker really turn between two frames?
A wheel with one painted marker turns f revolutions every second. The camera takes fs frames
every second, so the time between frames is 1/fs seconds. In that time the marker turns

    step = 360 * f / fs   degrees

Example: f = 28 rev/s and fs = 30 frames/s gives 360 * 28 / 30 = 336 degrees per frame.

In [ ]:
def step_angle_deg(f, fs):
    """True turn of the marker between two frames, in degrees (may exceed 360)."""
    return 360.0 * f / fs

## Step 2: what the eye sees
Your eye cannot know how many full turns happened between frames. It only sees where the
marker is now, and picks the shortest way from the old place to the new one. A turn of 336
degrees looks like a turn of 336 - 360 = -24 degrees (24 degrees *backwards*). So we fold the
step into the range -180 to +180 degrees.

In [ ]:
def fold_deg(angle):
    """Fold any angle (degrees) into [-180, 180): the shortest signed turn."""
    return (angle + 180.0) % 360.0 - 180.0


def apparent_frequency(f, fs):
    """Apparent turns per second (signed): negative means the wheel seems to go backwards."""
    return fold_deg(step_angle_deg(f, fs)) / 360.0 * fs


def direction(f, fs, tol=1e-9):
    a = apparent_frequency(f, fs)
    if abs(abs(a) - fs / 2.0) < tol:
        return "flicker"        # exactly half a turn per frame: no way to say which way
    if abs(a) < tol:
        return "frozen"
    return "forward" if a > 0 else "backwards"

## Step 3: the marker in every frame
`marker_angles` lists where the marker is (degrees, counter-clockwise from the right) in
frames 0, 1, 2, ... The picture the eye builds is the folded version of the same list.

In [ ]:
def marker_angles(f, fs, n_frames):
    k = np.arange(n_frames)
    return np.mod(step_angle_deg(f, fs) * k, 360.0)

## Step 4: see it
Left: the marker positions in the first frames for four wheel speeds filmed at 30 frames per
second. Right: apparent speed against true speed. The sawtooth is the whole story of this
repo: every 30 rev/s the picture starts again.

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    fs = 30.0
    cases = [3.0, 25.0, 29.0, 30.0]
    fig, axs = plt.subplots(1, 4, figsize=(10, 2.8))
    for ax, f in zip(axs, cases):
        ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, color="gray"))
        ang = np.radians(marker_angles(f, fs, 6))
        for k, a in enumerate(ang):
            ax.plot(np.cos(a), np.sin(a), "o", color=plt.cm.viridis(k / 6), ms=9)
            ax.text(1.25 * np.cos(a), 1.25 * np.sin(a), str(k), ha="center", va="center", fontsize=8)
        ax.set_xlim(-1.6, 1.6)
        ax.set_ylim(-1.6, 1.6)
        ax.set_aspect("equal")
        ax.axis("off")
        ax.set_title(f"true {f:.0f} rev/s\nseen {apparent_frequency(f, fs):+.0f} rev/s ({direction(f, fs)})", fontsize=9)
        print(f"true {f:4.1f} rev/s at {fs:.0f} fps -> step {step_angle_deg(f, fs):6.1f} deg, "
              f"seen {apparent_frequency(f, fs):+.2f} rev/s ({direction(f, fs)})")
    fig.suptitle("Frames 0 to 5 of a one-marker wheel filmed at 30 frames per second", fontsize=10)
    fig.savefig(OUT / "marker_frames.png", dpi=110, bbox_inches="tight")

    f_axis = np.linspace(0, 90, 901)
    seen = np.array([apparent_frequency(x, fs) for x in f_axis])
    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ax2.plot(f_axis, f_axis, ":", color="gray", label="true speed")
    ax2.plot(f_axis, seen, ".", ms=2, color="tab:blue", label="apparent speed")
    ax2.axhline(0, color="k", lw=0.6)
    ax2.set_xlabel("true wheel speed f (rev/s)")
    ax2.set_ylabel("apparent speed (rev/s)")
    ax2.set_title("Apparent speed at 30 frames per second")
    ax2.grid(alpha=0.3)
    ax2.legend()
    fig2.tight_layout()
    fig2.savefig(OUT / "apparent_vs_true.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Change `cases` in `lesson.py` to 10, 14, 16 and 20 rev/s at 30 frames per second. Predict the apparent speed of each before you run it.
2. **Stretch.** Write `frozen_speeds(fs, f_max)` that lists every wheel speed up to f_max that looks frozen at fs frames per second, and check it for fs = 24.
3. **Explain.** Explain why a wheel that turns exactly 15 rev/s at 30 fps cannot be said to go forward or backwards.

## Assessment
- **A. Conceptual.** A wheel turns slightly slower than fs rev/s (for example 29 rev/s at 30 fps). Why does it seem to turn slowly backwards, and not forward?
- **B. Numerical.** A wheel turns 33 rev/s and is filmed at 24 frames per second. Find the turn per frame, the apparent turn and the apparent speed. Do a sanity check.
- **C. Graph and simulation interpretation.** Using the sawtooth graph for 30 frames per second, for which true speeds between 0 and 60 rev/s does the wheel seem to go backwards, and where does it seem frozen?
- **D. Design and optimisation challenge.** A wheel turns 29 rev/s. You may film with 24, 25, 28, 29, 30 or 60 frames per second. Which choice shows it turning slowly forward at no more than 1 rev/s? Which choice shows its true speed? What does the true-speed choice cost?

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    checks.append(("worked example: 28 rev/s at 30 fps turns 336 degrees per frame, seen as -24",
                   abs(L.step_angle_deg(28, 30) - 336.0) < 1e-9 and abs(L.fold_deg(336.0) + 24.0) < 1e-9))
    checks.append(("worked example: 28 rev/s at 30 fps looks like -2 rev/s (backwards)",
                   abs(L.apparent_frequency(28, 30) + 2.0) < 1e-9 and L.direction(28, 30) == "backwards"))
    checks.append(("hand calculation: 33 rev/s at 24 fps is seen as +9 rev/s (135 degrees per frame)",
                   abs(L.fold_deg(L.step_angle_deg(33, 24)) - 135.0) < 1e-9 and abs(L.apparent_frequency(33, 24) - 9.0) < 1e-9))
    checks.append(("a slow wheel is seen correctly (3 rev/s at 30 fps)", abs(L.apparent_frequency(3, 30) - 3.0) < 1e-9))
    checks.append(("wheel turning exactly fs rev/s looks frozen", L.direction(30, 30) == "frozen" and abs(L.apparent_frequency(60, 30)) < 1e-9))
    checks.append(("exactly fs/2 is the flicker limit: |apparent| = 15 rev/s at 30 fps",
                   abs(abs(L.apparent_frequency(15, 30)) - 15.0) < 1e-9 and L.direction(15, 30) == "flicker"))
    seen = [L.apparent_frequency(x, 30) for x in np.linspace(0, 200, 401)]
    checks.append(("apparent speed never exceeds fs/2 in size", max(abs(s) for s in seen) <= 15.0 + 1e-9))
    checks.append(("speeds f and f + fs look the same", abs(L.apparent_frequency(7.0, 30) - L.apparent_frequency(37.0, 30)) < 1e-9))
    fr = L.marker_angles(25, 30, 4)
    checks.append(("25 rev/s at 30 fps: marker sits at 0, 300, 240, 180 degrees", np.allclose(fr, [0, 300, 240, 180])))
    checks.append(("design: 29 rev/s looks +1 rev/s at 28 fps, 0 at 29 fps, -1 at 30 fps",
                   abs(L.apparent_frequency(29, 28) - 1.0) < 1e-9 and L.direction(29, 29) == "frozen"
                   and abs(L.apparent_frequency(29, 30) + 1.0) < 1e-9))
    checks.append(("design: 29 rev/s looks +5, +4 and +29 rev/s at 24, 25 and 60 fps (60 fps is above 2 x 29 and shows the truth)",
                   [round(L.apparent_frequency(29, r), 9) for r in (24, 25, 60)] == [5.0, 4.0, 29.0]))
    checks.append(("a wheel at 25 rev/s filmed at 30 fps seems to turn at -5 rev/s: backwards", L.direction(25, 30) == "backwards"))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
